In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import math

import scipy
from scipy import stats

In [3]:
traindata = np.loadtxt("Data_OneStepAhead/Sunspot/train7.txt")
testdata = np.loadtxt("Data_OneStepAhead/Sunspot/test7.txt") 

In [4]:
# An example of a class
class Network:
    def __init__(self, Topo, Train, Test, MaxTime, MinPer):

        self.Top = Topo  # NN topology [input, hidden, output]
        self.Max = MaxTime  # max epocs
        self.TrainData = Train
        self.TestData = Test
        self.NumSamples = Train.shape[0]

        self.lrate = 0  # will be updated later with BP call

        self.minPerf = MinPer
        # initialize weights ( W1 W2 ) and bias ( b1 b2 ) of the network
        np.random.seed()
        self.W1 = np.zeros((self.Top[0], self.Top[1])  )
        self.B1 = np.zeros(self.Top[1])    # bias first layer
        self.W2 = np.zeros((self.Top[1], self.Top[2]) )
        self.B2 = np.zeros(self.Top[2])    # bias second layer
        self.hidout = np.zeros((self.Top[1]))  # output of first hidden layer
        self.out = np.zeros((self.Top[2]))  # output last layer

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sampleEr(self, actualout):
        error = np.subtract(self.out, actualout)
        sqerror = np.sum(np.square(error)) / self.Top[2]
        return sqerror

    def ForwardPass(self, X):
        z1 = X.dot(self.W1) - self.B1
        self.hidout = self.sigmoid(z1)  # output of first hidden layer#
        z2 = self.hidout.dot(self.W2) #- self.B2
        self.out = self.sigmoid(z2)  # output second hidden layer

    def BackwardPass(self, Input, desired):

        out_delta = (desired - self.out) * (self.out * (1 - self.out))
        hid_delta = np.zeros(self.Top[2])
        hid_delta = out_delta.dot(self.W2.T) * (self.hidout * (1 - self.hidout))

    # update weights and bias
        layer = 1  # hidden to output
        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W2[x, y] += self.lrate * out_delta[y] * self.hidout[x]
        for y in range(0, self.Top[layer + 1]):
            self.B2[y] += -1 * self.lrate * out_delta[y]

        layer = 0  # Input to Hidden

        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W1[x, y] += self.lrate * hid_delta[y] * Input[x]
        for y in range(0, self.Top[layer + 1]):
            self.B1[y] += -1 * self.lrate * hid_delta[y]


    def decode(self, w):
        w_layer1size = self.Top[0] * self.Top[1]
        w_layer2size =  self.Top[1] *  self.Top[2]

        w_layer1 = w[0:w_layer1size]
        self.W1 = np.reshape(w_layer1, ( self.Top[0],  self.Top[1]))

        w_layer2 = w[w_layer1size:w_layer1size + w_layer2size]
        self.W2 = np.reshape(w_layer2, ( self.Top[1],  self.Top[2]))
        self.B1 = w[w_layer1size + w_layer2size:w_layer1size + w_layer2size +  self.Top[1]]
        self.B2 = w[w_layer1size + w_layer2size + self.Top[1]:w_layer1size + w_layer2size + self.Top[1] + self.Top[2]]
    def encode(self):
        w1 = self.W1.ravel()
        w2 = self.W2.ravel()
        w = np.concatenate([w1, w2, self.B1, self.B2])
        return w
    def net_size(self):
        return ((self.Top[0] * self.Top[1]) + (self.Top[1] * self.Top[2]) + self.Top[1] + self.Top[2])
    def evaluate_proposal(self,   w ):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.
        size = self.TrainData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        fx = np.zeros(size)



        for pat in range(0, size):
            Input[:] =  self.TrainData[pat, 0:self.Top[0]]
            self.ForwardPass(Input)
            fx[pat] = self.out
        return fx

    def test_proposal(self,  w):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.

        size = self.TestData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        Desired = np.zeros((1, self.Top[2]))
        fx = np.zeros(size)

        sse = 0

        for pat in range(0, size):
            Input[:] = self.TestData[pat, 0:self.Top[0]]
            Desired[:] = self.TestData[pat, self.Top[0]:]
            self.ForwardPass(Input)
            fx[pat] = self.out
            sse = sse + self.sampleEr(Desired)

        rmse = np.sqrt(sse / size)


        return [fx,rmse]


In [5]:
input = 7  # max input
hidden = 5
output = 1
baseNet = [input, hidden, output]
secondnet = [input, hidden, output]
mtaskNet = np.array([baseNet, secondnet])
dims = len(mtaskNet)

In [6]:
Net = Network(mtaskNet[0], traindata, testdata, 0.1, 0.0002)
netsize = Net.net_size()
w = np.random.randn(netsize)
print(Net.evaluate_proposal(w))

[0.39570458 0.39613727 0.3962076  0.39889393 0.39966174 0.40228709
 0.40752883 0.41256577 0.42311045 0.433785   0.44765294 0.46560874
 0.48374224 0.50246815 0.51499481 0.52640145 0.54307423 0.55367033
 0.56004622 0.56835352 0.57200094 0.57135212 0.56633778 0.56006454
 0.557043   0.54816204 0.54131281 0.53487407 0.52576992 0.5190375
 0.50647272 0.498192   0.49762917 0.49469125 0.49409492 0.49386362
 0.49302463 0.48942583 0.48124689 0.47531796 0.46537697 0.45483805
 0.44908224 0.44259695 0.43679475 0.431767   0.42772791 0.42397781
 0.41898181 0.41534159 0.41203404 0.41151474 0.40899598 0.40742066
 0.40593173 0.40438016 0.40192812 0.3991371  0.398783   0.39909595
 0.39866715 0.39951087 0.40046952 0.40064141 0.4027867  0.40635991
 0.41053163 0.41331084 0.41712706 0.42176492 0.42560025 0.42938656
 0.43344068 0.4378946  0.44557554 0.45076721 0.45630305 0.46061942
 0.46218531 0.46566663 0.47574227 0.49129589 0.50784092 0.5236337
 0.53617617 0.54835802 0.55032273 0.54852367 0.55167022 0.549935

In [7]:
Net1 = Network(mtaskNet[1], traindata, testdata, 0.1, 0.0002)
netsize1 = Net1.net_size()
w_pro = np.random.randn(netsize1)
w_pro

array([-1.2231587 , -1.21402097,  0.02699955, -0.49191418, -1.61417567,
       -0.21283402,  3.33390736, -0.17079047,  1.32486393,  0.22548742,
        2.0278812 , -0.67979687, -0.46488753, -0.60074856, -1.35681476,
       -0.6482227 , -0.62168817, -0.34082022,  0.24074531,  1.96873756,
       -0.40384535, -1.30079838, -1.24449405,  1.28745023, -1.00794201,
        1.54076216,  1.28410984, -0.27036689,  1.37471911,  0.04102154,
        0.56322076, -1.56088406,  0.65162383,  1.23299911,  0.53963614,
        0.98095189, -0.10262825,  0.18900671, -0.3878844 ,  1.4080113 ,
       -2.35813568,  0.54203844, -0.32916097,  0.33727907,  0.53532366,
       -1.25212067])

In [8]:
#propose for w
Netlist = [None] * 2
netsize = []
for n in range(0, dims):
    Netlist[n] = Network(mtaskNet[n], traindata, testdata, 0.1, 0.0002)
    netsize.append(Netlist[n].net_size())
netsize

[46, 46]

In [9]:
for n in range(1, dims):
    shape_diff = netsize[n] - netsize[n-1]
    print(shape_diff)
    #w_pro[s, :netsize[s]] = w[s, :netsize[s]] + np.random.normal(0, step_w, netsize[s])
    u = np.random.normal(0, 1, shape_diff)
    w_pro[n, :netsize[n]] = np.lib.pad(w[n-1,:netsize[n-1]], (0, shape_diff), 'constant', constant_values=(0,u))
    print(np.array(w_pro[n, :netsize[n]]))
    print(np.random.rand(netsize[n]))

0


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [11]:
class MCMC:
	def __init__(self, samples, traindata, testdata, minPerf, mtaskNet):
		self.samples = samples  
		self.mtaskNet = mtaskNet  # mtaskNet = np.array([baseNet, secondnet]) -> to access each nn, mtaskNet[dims]
		self.traindata = traindata  #
		self.testdata = testdata 
		self.minPerf = minPerf
		# ----------------

	def rmse(self, predictions, targets):
		return np.sqrt(((predictions - targets) ** 2).mean())

	def likelihood_func(self, neuralnet, data, w, tausq, dim):
		topology = self.mtaskNet[dim]
		y = data[:, topology[0]]
		fx = neuralnet.evaluate_proposal(w)
		rmse = self.rmse(fx, y)
		n = y.shape[0]

		loss =( -(n/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(y - fx))) #tausq is variance of error terms
		return [loss, fx, rmse]

	def prior_likelihood(self, sigma_squared, nu_1, nu_2, w, tausq):
		h = self.topology[1]  # number hidden neurons
		d = self.topology[0]  # number input neurons
		part1 = -1 * ((d * h + h + 2) / 2) * np.log(sigma_squared)
		part2 = 1 / (2 * sigma_squared) * (sum(np.square(w)))
		log_loss = part1 - part2  - (1 + nu_1) * np.log(tausq) - (nu_2 / tausq)
		return log_loss
	
	def dim_jump(self, netw_prev): #to jump to larger dim
		netw = self.mtaskNet[self.dim]
		w_size = (netw[0] * netw[1]) + (netw[1] * netw[2]) + netw[1] + netw[2]
		w_size_prev = (netw_prev[0] * netw_prev[1]) + (netw_prev[1] * netw_prev[2]) + netw_prev[1] + netw_prev[2]
		shape_diff = w_size - w_size_prev
		w_prev = np.random.randn(w_size_prev)
		v = np.random.normal(0, 1, shape_diff)
		w = np.lib.pad(w_prev, (0, shape_diff), 'constant', constant_values=(0,v))
		return w #current

	def sampler(self, w_limit, tau_limit):

		# ------------------- initialize MCMC
		testsize = self.testdata.shape[0]
		trainsize = self.traindata.shape[0]
		samples = self.samples


		self.sgd_depth = 1

		x_test = np.linspace(0, 1, num=testsize)
		x_train = np.linspace(0, 1, num=trainsize)

		netw = self.mtaskNet[self.dim]  # choose a netw from mtaskNet given dim [input, hidden, output]
		y_test = self.testdata[:, netw[0]]
		y_train = self.traindata[:, netw[0]]


		w_size = (netw[0] * netw[1]) + (netw[1] * netw[2]) + netw[1] + netw[2]  # num of weights and bias

		pos_w = np.ones((samples, w_size))  # posterior of all weights and bias over all samples
		pos_tau = np.ones((samples, 1))

		fxtrain_samples = np.ones((samples, trainsize))  # fx of train data over all samples
		fxtest_samples = np.ones((samples, testsize))  # fx of test data over all samples
		rmse_train = np.zeros(samples)
		rmse_test = np.zeros(samples)

		w = self.dim_jump(self.mtaskNet[self.dim - 1]) #jump from prev w to current w with w_size
		w_proposal = np.random.randn(w_size)

		step_w = w_limit  # defines how much variation you need in changes to w
		step_eta = tau_limit #exp 1
		# --------------------- Declare FNN and initialize
		 
		neuralnet = Network(self.mtaskNet[self.dim], self.traindata, self.testdata, 5, self.minPerf)
		print('evaluate Initial w')

		pred_train = neuralnet.evaluate_proposal(w)
		pred_test = neuralnet.test_proposal(w)

		eta = np.log(np.var(pred_train - y_train))
		tau_pro = np.exp(eta)

		sigma_squared = 25
		nu_1 = 0
		nu_2 = 0
 
#--------
		sigma_diagmat = np.zeros((w_size, w_size))  # for Equation 9 in Ref [Chandra_ICONIP2017]
		np.fill_diagonal(sigma_diagmat, step_w)
#--------------
		prior_current = self.prior_likelihood(sigma_squared, nu_1, nu_2, w, tau_pro)  # tau pro is variance of noise

		[likelihood, pred_train, rmsetrain] = self.likelihood_func(neuralnet, self.traindata, w, tau_pro)
		[likelihood_ignore, pred_test, rmsetest] = self.likelihood_func(neuralnet, self.testdata, w, tau_pro)

		print(likelihood, ' Initial likelihood')

		naccept = 0

		for i in range(samples - 1):

			w_proposal = np.random.normal(w, step_w, w_size)

 			
			eta_pro = eta + np.random.normal(0, step_eta, 1)
			tau_pro = math.exp(eta_pro)

			[likelihood_proposal, pred_train, rmsetrain] = self.likelihood_func(neuralnet, self.traindata, w_proposal,
																				tau_pro)
			[likelihood_ignore, pred_test, rmsetest] = self.likelihood_func(neuralnet, self.testdata, w_proposal,
																			tau_pro) 

			prior_prop = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_proposal,
											   tau_pro)  # takes care of the gradients


			diff_prior = prior_prop - prior_current

			diff_likelihood = likelihood_proposal - likelihood

			#mh_prob = min(1, math.exp(diff_likelihood + diff_prior + diff_prop))

			try:
				mh_prob = min(1, math.exp(diff_likelihood+diff_prior))

			except OverflowError as e:
				mh_prob = 1



			u = random.uniform(0, 1)

			if u < mh_prob:
				# Update position 
				naccept += 1
				likelihood = likelihood_proposal
				prior_current = prior_prop
				w = w_proposal
				eta = eta_pro
				if i%10 ==0:
					print(i,likelihood, prior_current, rmsetrain, rmsetest, 'accepted')
				 

				pos_w[i + 1,] = w_proposal
				pos_tau[i + 1,] = tau_pro
				fxtrain_samples[i + 1,] = pred_train
				fxtest_samples[i + 1,] = pred_test
				rmse_train[i + 1,] = rmsetrain
				rmse_test[i + 1,] = rmsetest

				plt.plot(x_train, pred_train)


			else:
				pos_w[i + 1,] = pos_w[i,]
				pos_tau[i + 1,] = pos_tau[i,]
				fxtrain_samples[i + 1,] = fxtrain_samples[i,]
				fxtest_samples[i + 1,] = fxtest_samples[i,]
				rmse_train[i + 1,] = rmse_train[i,]
				rmse_test[i + 1,] = rmse_test[i,]
 

		print(naccept, ' num accepted')
		print(naccept / (samples * 1.0), '% was accepted')
		accept_ratio = naccept / (samples * 1.0) * 100

		return (pos_w, pos_tau, fxtrain_samples, fxtest_samples, x_train, x_test, rmse_train, rmse_test, accept_ratio)


In [12]:
def likelihood_func(neuralnet, data, w, tausq, dims):
		topology = mtaskNet[dims]
		y = data[:, topology[0]]
		fx = neuralnet.evaluate_proposal(w)
		#rmse = rmse(fx, y)
		n = y.shape[0]

		loss =( -(n/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(y - fx))) #tausq is variance of error terms
		return [loss, fx]

In [13]:
loss, fx = likelihood_func(Net, traindata, w, 25, 0)

In [14]:
loss1, fx1 = likelihood_func(Net1, traindata, w_pro, 25, 1)
print(loss, loss1)

-751.2235202874112 -752.2508784901256


In [15]:
def prior_likelihood(sigma_squared, nu_1, nu_2, w, tausq, model_prior, dims):
    topology = mtaskNet[dims]
    h = topology[1]  # number hidden neurons
    d = topology[0]  # number input neurons
    part1 = -1 * ((d * h + h + 2) / 2) * np.log(sigma_squared)
    part2 = -1 / (2 * sigma_squared) * (sum(np.square(w)))
    part3 = 1 * np.log(model_prior)
    log_loss = part1 + part2 + part3 - (1 + nu_1) * np.log(tausq) - (nu_2 / tausq)
    return log_loss

In [16]:
log_loss = prior_likelihood(25, 10, 10, w, 25, 1/2, 0)
log_loss1 = prior_likelihood(25, 10, 10, w_pro, 25, 1/2, 1)
print(log_loss, log_loss1)

-105.06767033580823 -105.27297460896378


In [ ]:
def dim_diff(dims1, dims2): #to jump dims1= 0, dims2= 1
    netw_1 = mtaskNet[dims1]
    w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
    netw_2 = mtaskNet[dims2]
    w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]
    shape_diff = max(w_size1 - w_size2, w_size2 - w_size1)
    return shape_diff

In [20]:
#shape_diff = dim_diff(0, 1)
v = np.random.normal(0, 1, 9)
shape_diff
v

array([ 2.15794018,  0.27826344, -1.26935563,  0.28987312,  0.12652567,
       -0.25872303, -1.58473326,  0.40420541,  1.2427719 ])

In [18]:
def v_jump(tausq, v, dims1, dims2):
    n = 9

    log_v =( -(n/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(v))) #tausq is variance of error terms
    return log_v

In [21]:
v_jump(25, v, 0, 1)

-22.970020337468753

In [22]:
def sampler(samples, dims1, dims2):
    netw_1 = mtaskNet[dims1]
    w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
    netw_2 = mtaskNet[dims2]
    w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]
    w = np.random.rand(w_size1)
    w_dim2 = np.random.rand(w_size2)
    
    [likelihood, pred_train] = likelihood_func(Net, traindata, w, 25, dims1) #w here used for bnn
    prior = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)
    naccept = 0
    for i in range(samples -1):
        w_proposal = np.lib.pad(w, (0, shape_diff), 'constant', constant_values=(0,v))
        [likelihood_proposal, pred_train_proposal] = likelihood_func(Net1, traindata, w_proposal, 25, dims2)
        prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal, 25, 1/2, dims2)
        log_v = v_jump(25, v, 0, 1)
        log_posterior = (prior_proposal + likelihood_proposal) - (prior + likelihood + log_v) 
        try:
            mh_prob = min(1, math.exp(log_posterior))

        except OverflowError as e:
            mh_prob = 1

        u = random.uniform(0, 1)

        if u < mh_prob:
            # ACCEPT
            naccept += 1
            if w_proposal.shape[0] == w_dim2.shape[0]:
                likelihood = likelihood_proposal
                prior = prior_proposal            
                w = w_proposal #assign proposed w to w which now has higher dim 
            else:
                w_proposal_down = w_proposal[0:w_size1] #propose w_down that has lower dim
                [likelihood_proposal_down, pred_train_proposal_down] = likelihood_func(Net, traindata, w_proposal_down, 25, dims1)
                prior_proposal_down = prior_likelihood(25, 0.1, 0.2, w_proposal_down, 25, 1/2, dims1)
                log_posterior_down = (prior_proposal_down + likelihood_proposal_down + log_v) - (prior_proposal + likelihood_proposal) 
                try:
                    mh_prob_down = min(1, math.exp(log_posterior_down))
                except OverflowError as e:
                    mh_prob_down = 1
                u_down = random.uniform(0, 1)
                if u_down < mh_prob_down:
                    naccept += 1
                    likelihood_proposal = likelihood_proposal_down
                    prior_proposal = prior_proposal_down
                    w = w_proposal_down
            print(i,likelihood, prior, w, w.shape[0], 'accepted')


In [ ]:
random.choice([-1, 1])

-1

In [23]:
#in this case, mtaskNet = [baseNet, baseNet]
def sampler(samples, dims1, dims2):
    netw_1 = mtaskNet[dims1]
    w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
    netw_2 = mtaskNet[dims2]
    w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]
    w = np.zeros(w_size1)
    w_dim2 = np.zeros(w_size2)
    W = []
    naccept = 0
    for i in range(samples -1):
        netw_2[1] = netw_1[1] + random.choice([-1, 1])
        if w_size2 > w_size1:
            w = np.random.rand(w_size1)
            [likelihood_dim1, pred_train] = likelihood_func(Net, traindata, w, 25, dims1) #w here used for bnn
            prior_dim1 = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)
            w_proposal = np.lib.pad(w, (0, shape_diff), 'constant', constant_values=(0,v))
            [likelihood_proposal, pred_train_proposal] = likelihood_func(Net1, traindata, w_proposal, 25, dims2)
            prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal, 25, 1/2, dims2)
            log_v = v_jump(25, v, 0, 1)
            log_posterior = (prior_proposal + likelihood_proposal) - (prior_dim1 + likelihood_dim1 + log_v) 
            try:
                mh_prob = min(1, math.exp(log_posterior))

            except OverflowError as e:
                mh_prob = 1

            u = random.uniform(0, 1)

            if u < mh_prob:
                # ACCEPT
                naccept += 1
                likelihood_dim1 = likelihood_proposal
                prior_dim1 = prior_proposal            
                w_dim2 = w_proposal #assign proposed w to w which now has higher dim
                W.append(w_dim2)
        else:
            w_dim2 = np.random.rand(w_size2)
            [likelihood_dim2, pred_train] = likelihood_func(Net1, traindata, w_dim2, 25, dims2) #w here used for bnn
            prior_dim2 = prior_likelihood(25, 0.1, 0.2, w_dim2, 25, 1/2, dims2)
            w_proposal_down = w_dim2[0:w_size1] #propose w_down that has lower dim
            [likelihood_proposal_down, pred_train_proposal_down] = likelihood_func(Net, traindata, w_proposal_down, 25, dims1)
            prior_proposal_down = prior_likelihood(25, 0.1, 0.2, w_proposal_down, 25, 1/2, dims1)
            log_posterior_down = (prior_proposal_down + likelihood_proposal_down + log_v) - (prior_dim2 + likelihood_dim2) 
            try:
                mh_prob_down = min(1, math.exp(log_posterior_down))
            except OverflowError as e:
                mh_prob_down = 1
            u_down = random.uniform(0, 1)
            if u_down < mh_prob_down:
                naccept += 1
                likelihood_dim2 = likelihood_proposal_down
                prior_dim2 = prior_proposal_down
                w = w_proposal_down
                W.append(w)
        print(i,W, 'accepted')


In [24]:
sampler(10, 0, 1)

ValueError: cannot reshape array of size 4 into shape (6,1)

In [37]:
#in this case, mtaskNet = [baseNet, baseNet]
def sampler(samples, dims1, dims2):
    netw_1 = mtaskNet[dims1]
    netw_2 = mtaskNet[dims2]
    #w = np.zeros(w_size1)
    #w_dim2 = np.zeros(w_size2)
    Net1 = Network(mtaskNet[dims1], traindata, testdata, 0.1, 0.0002)
    Net2 = Network(mtaskNet[dims2], traindata, testdata, 0.1, 0.0002)

    W = []
    naccept = 0
    for i in range(samples -1):
        w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
        netw_2[1] = netw_1[1] + random.choice([-1, 1])
        w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]

        if w_size2 > w_size1:
            #pos_w = np.ones((samples, w_size))
            w = np.random.rand(w_size1)
            [likelihood_current, pred_train] = likelihood_func(Net1, traindata, w, 25, dims1) #w here used for bnn
            prior_current = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)

            v = np.random.normal(0, 1, w_size2-w_size1)
            w_proposal = np.lib.pad(w, (0, w_size2-w_size1), 'constant', constant_values=(0,v))
            [likelihood_proposal, pred_train_proposal] = likelihood_func(Net2, traindata, w_proposal, 25, dims2)
            prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal, 25, 1/2, dims2)


            log_v = v_jump(25, v, 0, 1)
            log_posterior = (prior_proposal + likelihood_proposal) - (prior_current + likelihood_current + log_v) 
            try:
                mh_prob = min(1, math.exp(log_posterior))

            except OverflowError as e:
                mh_prob = 1

            u = random.uniform(0, 1)

            if u < mh_prob:
                # ACCEPT
                naccept += 1
                likelihood_current = likelihood_proposal
                prior_current = prior_proposal            
                w = w_proposal #assign proposed w to w which now has higher dim
                W.append(w)

        else: #w_size2 < w_size1
            w = np.random.rand(w_size1)
            [likelihood_current, pred_train] = likelihood_func(Net1, traindata, w, 25, dims1) #w here used for bnn
            prior_current = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)

            w_proposal_down = w[0:w_size2] #propose w_down that has lower dim
            [likelihood_proposal, pred_train_proposal] = likelihood_func(Net2, traindata, w_proposal_down, 25, dims2)
            prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal_down, 25, 1/2, dims2)

            v = np.random.normal(0, 1, w_size1-w_size2)
            log_v = v_jump(25, v, 0, 1)
            log_posterior_down = (prior_proposal + likelihood_proposal + log_v) - (prior_current + likelihood_current) 
            try:
                mh_prob_down = min(1, math.exp(log_posterior_down))
            except OverflowError as e:
                mh_prob_down = 1
            u_down = random.uniform(0, 1)
            if u_down < mh_prob_down:
                naccept += 1
                likelihood_current = likelihood_proposal
                prior_current = prior_proposal
                w = w_proposal_down
                W.append(w)
        accept_ratio = naccept / (samples)
        print(i, 'jump from model 1 with size', w_size1,'to model 2 with size', w_size2, np.shape(W), '{:.1%}'.format(accept_ratio), 'is accepted')
    return W, naccept


0 jump from model 1 with size 46 to model 2 with size 37 (0,) 0.0% is accepted
1 jump from model 1 with size 46 to model 2 with size 55 (1, 55) 20.0% is accepted
2 jump from model 1 with size 46 to model 2 with size 37 (1, 55) 20.0% is accepted
3 jump from model 1 with size 46 to model 2 with size 37 (1, 55) 20.0% is accepted


/anaconda3/envs/env_pytorch/lib/python3.6/site-packages/numpy/lib/arraypad.py:490: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  x = np.array(x)


[array([ 0.44797142,  0.30023736,  0.81492668,  0.84382933,  0.22073255,
         0.31223454,  0.59751666,  0.30968658,  0.14764358,  0.07633811,
         0.44289947,  0.40240071,  0.89927509,  0.37622035,  0.25064564,
         0.98816349,  0.44494953,  0.2504649 ,  0.53553218,  0.13790369,
         0.32294258,  0.83980188,  0.68311534,  0.82056364,  0.2099674 ,
         0.86603082,  0.09156325,  0.3261167 ,  0.72276004,  0.33029376,
         0.02685687,  0.14298997,  0.76929122,  0.98050643,  0.17892468,
         0.66109859,  0.91035815,  0.5259921 ,  0.89660343,  0.04554258,
         0.85918148,  0.15818909,  0.37030843,  0.43506449,  0.20364548,
         0.44951967,  0.37304211,  1.67106634,  0.41089561, -0.4114788 ,
        -0.56223023,  0.35888423, -0.67005963, -0.25525672, -0.05482876])]

In [39]:
#in this case, mtaskNet = [baseNet, baseNet]
def sampler(samples, dims1, dims2):
    netw_1 = mtaskNet[dims1]
    netw_2 = mtaskNet[dims2]
    netw = [netw_1[1], netw_2[1]]
    #w = np.zeros(w_size1)
    #w_dim2 = np.zeros(w_size2)
    Net1 = Network(mtaskNet[dims1], traindata, testdata, 0.1, 0.0002)
    Net2 = Network(mtaskNet[dims2], traindata, testdata, 0.1, 0.0002)

    W = []
    naccept = 0
    for i in range(samples -1):
        netw_2[1] = netw_1[1] + random.choice([-1, 1])
        random.shuffle(netw)
        # Assign new hidden neurons
        netw_1[1] = netw[0]
        netw_2[1] = netw[1]
        w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
        w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]

        if w_size2 > w_size1:
            #pos_w = np.ones((samples, w_size))
            w = np.random.rand(w_size1)
            [likelihood_current, pred_train] = likelihood_func(Net1, traindata, w, 25, dims1) #w here used for bnn
            prior_current = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)

            v = np.random.normal(0, 1, w_size2-w_size1)
            w_proposal = np.lib.pad(w, (0, w_size2-w_size1), 'constant', constant_values=(0,v))
            [likelihood_proposal, pred_train_proposal] = likelihood_func(Net2, traindata, w_proposal, 25, dims2)
            prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal, 25, 1/2, dims2)


            log_v = v_jump(25, v, 0, 1)
            log_posterior = (prior_proposal + likelihood_proposal) - (prior_current + likelihood_current + log_v) 
            try:
                mh_prob = min(1, math.exp(log_posterior))

            except OverflowError as e:
                mh_prob = 1

            u = random.uniform(0, 1)

            if u < mh_prob:
                # ACCEPT
                naccept += 1
                likelihood_current = likelihood_proposal
                prior_current = prior_proposal            
                w = w_proposal #assign proposed w to w which now has higher dim
                W.append(w)

        else: #w_size2 < w_size1
            w = np.random.rand(w_size1)
            [likelihood_current, pred_train] = likelihood_func(Net1, traindata, w, 25, dims1) #w here used for bnn
            prior_current = prior_likelihood(25, 0.1, 0.2, w, 25, 1/2, dims1)

            w_proposal_down = w[0:w_size2] #propose w_down that has lower dim
            [likelihood_proposal, pred_train_proposal] = likelihood_func(Net2, traindata, w_proposal_down, 25, dims2)
            prior_proposal = prior_likelihood(25, 0.1, 0.2, w_proposal_down, 25, 1/2, dims2)

            v = np.random.normal(0, 1, w_size1-w_size2)
            log_v = v_jump(25, v, 0, 1)
            log_posterior_down = (prior_proposal + likelihood_proposal + log_v) - (prior_current + likelihood_current) 
            try:
                mh_prob_down = min(1, math.exp(log_posterior_down))
            except OverflowError as e:
                mh_prob_down = 1
            u_down = random.uniform(0, 1)
            if u_down < mh_prob_down:
                naccept += 1
                likelihood_current = likelihood_proposal
                prior_current = prior_proposal
                w = w_proposal_down
                W.append(w)
        accept_ratio = naccept / (samples)
        print(i, 'jump from model 1 with size', w_size1,'to model 2 with size', w_size2, np.shape(W), '{:.1%}'.format(accept_ratio), 'is accepted')
    return W, naccept

In [32]:
sampler(10, 0, 1)

0 jump from model 1 with size 46 to model 2 with size 37 (0,) 0.0% is accepted
1 jump from model 1 with size 46 to model 2 with size 37 (0,) 0.0% is accepted
2 jump from model 1 with size 37 to model 2 with size 46 (1, 46) 10.0% is accepted
3 jump from model 1 with size 37 to model 2 with size 46 (2, 46) 20.0% is accepted
4 jump from model 1 with size 37 to model 2 with size 46 (3, 46) 30.0% is accepted
5 jump from model 1 with size 37 to model 2 with size 46 (4, 46) 40.0% is accepted
6 jump from model 1 with size 37 to model 2 with size 46 (5, 46) 50.0% is accepted
7 jump from model 1 with size 37 to model 2 with size 46 (6, 46) 60.0% is accepted
8 jump from model 1 with size 46 to model 2 with size 37 (6, 46) 60.0% is accepted
